<div style="max-width:100%;box-sizing:border-box;border-top:4px solid #0f766e;padding:24px 0">
<div style="color:#0f766e;font-weight:700">EXTENSION · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>Module 4–5 Extension Lab: Direct File Queries, Batch Loading, and Group Commit Acknowledgments</h1>
<p>Target Doris 4.1.3 · Independent ext_* lab tables</p>
</div>

[Extension home](README.md) · [Main course contents](../README.md)

Complete Labs 1, 4, and 5 first, using the same course sandbox and MinIO. This lab checks and prepares the course lake table environment without connecting to external production services.
Rebuild only `ext_file_insert`, `ext_file_broker`, `ext_defaults`, and `ext_gc_off_mode/sync_mode/async_mode`.
Only add random file paths under the current lab database in object storage; do not delete other files. Allow 30–45 minutes, independent of the main labs.
Run only one kernel at a time. On failure, inspect the original load response and SHOW LOAD first; do not immediately resend an asynchronous load.

Validation: direct queries of the same WWI ten-order file and both loaded results match row by row; defaults/generated columns can be verified; all three acknowledgment modes ultimately produce 10 rows totaling 1400.00.
This is not a Kafka, CDC, or continuous file discovery lab, and it does not verify recovery from WAL disk failures.

In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import dataset_path, expect, fixture, normalized
from dw_course.ui import show_sql, show_records, show_response
lab = connect_sandbox()
lab.sql("SELECT VERSION() AS version")
lab.sql("SHOW BACKENDS")

## 1. Write the Ten-Order Sample to a Separate Parquet File

This creates a standalone file, not an Iceberg table. The file uses the same ten-order sample as the Lab 4 lake table.
Python accesses host port 51900; Doris accesses course-lake-minio:9000 over the course network. Do not mix them up.

In [ ]:
from dw_course.lakehouse import prepare_lakehouse, ACCESS_KEY, SECRET_KEY
from dw_course.wwi import history_rows, history_ddl, HISTORY_COLUMNS
from decimal import Decimal
from datetime import date
from uuid import uuid4
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
lake_table = prepare_lakehouse(lab, start=True)
records = [dict(zip(HISTORY_COLUMNS, row)) for row in history_rows()]
for row in records:
    row["order_amount"] = Decimal(row["order_amount"])
    row["order_date"] = date.fromisoformat(row["order_date"])
buffer = pa.BufferOutputStream()
pq.write_table(pa.Table.from_pylist(records), buffer)
key = f"{lab.database}/file_demo/{uuid4().hex}/orders.parquet"
s3 = boto3.client("s3",endpoint_url="http://127.0.0.1:51900",aws_access_key_id=ACCESS_KEY,aws_secret_access_key=SECRET_KEY,region_name="us-east-1")
s3.put_object(Bucket="course-warehouse",Key=key,Body=buffer.getvalue().to_pybytes())
uri = "s3://course-warehouse/" + key
tvf = f'''S3("uri"="{uri}","format"="parquet",
"s3.endpoint"="http://course-lake-minio:9000","s3.region"="us-east-1",
"s3.access_key"="{ACCESS_KEY}","s3.secret_key"="{SECRET_KEY}","use_path_style"="true")'''
projection = "order_id,customer_id,CAST(order_date AS STRING),order_amount,line_count,data_source"
expect(lab.query(f"SELECT {projection} FROM {tvf} ORDER BY order_id"), history_rows())
expect(lab.query(f"SELECT {projection} FROM {lake_table} ORDER BY order_id"), history_rows())
lab.sql(f"SELECT COUNT(*) AS orders,SUM(order_amount) AS amount FROM {tvf}")

## 2. Synchronous INSERT SELECT and Asynchronous Broker Load

The two methods write to separate tables. `WITH S3` uses the S3 protocol and needs no additional Broker process.
After successful Broker Load submission, wait for `State=FINISHED`; cancellation or timeout means failure, so do not continue reporting success.
A timeout requests cancellation of this unique label without resetting the entire environment. Failure details remain in SHOW LOAD.

In [ ]:
import time
for table in ("ext_file_insert", "ext_file_broker"):
    lab.execute("DROP TABLE IF EXISTS " + table)
    lab.execute(history_ddl(table))
lab.execute(f"INSERT INTO ext_file_insert ({','.join(HISTORY_COLUMNS)}) SELECT {','.join(HISTORY_COLUMNS)} FROM {tvf}")
label = "ext_broker_" + uuid4().hex
statement = f'''LOAD LABEL {lab.database}.{label} (
DATA INFILE("{uri}") INTO TABLE ext_file_broker FORMAT AS "parquet"
({','.join(HISTORY_COLUMNS)})
) WITH S3 (
"AWS_ENDPOINT"="http://course-lake-minio:9000","AWS_REGION"="us-east-1",
"AWS_ACCESS_KEY"="{ACCESS_KEY}","AWS_SECRET_KEY"="{SECRET_KEY}","use_path_style"="true"
) PROPERTIES ("timeout"="120","max_filter_ratio"="0")'''
show_sql("Submit batch job", statement)
lab.execute(statement)
deadline = time.monotonic()+150
while True:
    rows = lab.query("SHOW LOAD WHERE Label = %s", (label,))
    if rows and rows[0][2] == "FINISHED":
        break
    if rows and rows[0][2] == "CANCELLED":
        raise RuntimeError(f"Broker Load cancelled: {rows}")
    if time.monotonic() >= deadline:
        lab.execute("CANCEL LOAD WHERE LABEL = %s", (label,))
        raise TimeoutError(f"Broker Load timeout: {rows}")
    time.sleep(1)
lab.sql("SHOW LOAD WHERE Label = %s", (label,))
for table in ("ext_file_insert", "ext_file_broker"):
    expect(lab.query(f"SELECT {projection} FROM {table} ORDER BY order_id"), history_rows())

## 3. Defaults and Generated Columns Are Different

Omitting status supplies CREATED; explicitly specifying PAID preserves the input.
amount is generated from amount_cents: 18000 fen → 180.00 yuan. INSERT does not supply the generated column.
These two records are independent teaching samples and are not appended to history or current-state tables.

In [ ]:
lab.execute("DROP TABLE IF EXISTS ext_defaults")
lab.execute('''CREATE TABLE ext_defaults (
order_id BIGINT, amount_cents BIGINT NOT NULL,
status VARCHAR(20) NOT NULL DEFAULT "CREATED",
amount DECIMAL(12,2) AS (CAST(amount_cents AS DECIMAL(12,2)) / 100)
) DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_defaults (order_id,amount_cents) VALUES (901001,18000)")
lab.execute("INSERT INTO ext_defaults (order_id,amount_cents,status) VALUES (901002,8000,'PAID')")
expect(lab.query("SELECT order_id,status,amount FROM ext_defaults ORDER BY order_id"), [(901001,"CREATED","180.00"),(901002,"PAID","80.00")])
lab.sql("SELECT * FROM ext_defaults ORDER BY order_id")

## 4. Group Commit Response Time Is Not Query Visibility Time

Send the same ten simulated orders to three empty tables separately, recording when the response arrives and when all data first becomes queryable.
Group Commit requests do not specify a custom label; confirm the path through the returned GroupCommit field rather than applying off_mode label-retry conclusions.
Observe whether data is already visible on the first async query; do not require it to be "definitely invisible" or assume a fixed latency ratio.
This lab compares only acknowledgment and visibility; whether multiple requests share a transaction, sustained throughput, and version backlogs require a separate concurrent-load experiment.

In [ ]:
import os
import requests
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
payload = (dataset_path("orders.csv")).read_bytes()
endpoint = os.environ["DW_BE_HTTP_URL"].rstrip("/")
measurements = []
for mode in ("off_mode", "sync_mode", "async_mode"):
    table = "ext_gc_" + mode
    lab.execute("DROP TABLE IF EXISTS " + table)
    lab.execute(order_ddl(table))
    lab.execute(f'ALTER TABLE {table} SET ("group_commit_interval_ms"="2000")')
    headers = {"format":"csv", "column_separator":",", "columns":",".join(ORDER_COLUMNS),
               "strict_mode":"true", "max_filter_ratio":"0", "group_commit":mode}
    if mode == "off_mode":
        headers["label"] = "ext_off_" + uuid4().hex
    started = time.monotonic()
    response = requests.put(f"{endpoint}/api/{lab.database}/{table}/_stream_load",
                            auth=(lab.user,lab.password),headers=headers,data=payload,
                            allow_redirects=False,timeout=120)
    response.raise_for_status()
    result = response.json()
    show_response(result, title=mode + " Stream Load response")
    expect(result["Status"], "Success")
    expect(result["NumberLoadedRows"], 10)
    if mode != "off_mode":
        expect(result["GroupCommit"], True)
    acknowledged = time.monotonic()
    first_count = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    count = first_count
    while count != 10:
        if count > 10:
            raise RuntimeError(f"Unexpected duplicate rows: {count}")
        if time.monotonic()-acknowledged > 60:
            raise TimeoutError(f"Data not visible: {result}")
        time.sleep(0.1)
        count = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    visible = time.monotonic()
    projection = ",".join("DATE_FORMAT(event_time,'%Y-%m-%d %H:%i:%s')" if col=="event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM {table} ORDER BY order_id"), order_rows(fixture("orders.json")))
    measurements.append({"Mode": mode, "Response ms": round((acknowledged-started)*1000,2),
                         "First full visibility ms": round((visible-started)*1000,2),
                         "Rows in first query after response": first_count})
    lab.sql(f"SHOW TABLETS FROM {table}")
show_records("Group Commit comparison", measurements)

## 5. Independent Explanation and Troubleshooting

Explain why direct file queries do not need CREATE CATALOG while Iceberg does, and why LOAD LABEL returning does not immediately mean completion.
Then explain when defaults and generated columns take effect, and why an async success response alone does not prove the data is queryable.
On failure, retain response, label, TxnId, and SHOW LOAD; for a Group Commit timeout, investigate the WAL/jobs before blindly resending.

References: [S3 TVF](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/s3/),
[Broker Load](https://doris.apache.org/docs/4.x/data-operate/import/import-way/broker-load-manual/),
[Group Commit](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/group-commit-manual/),
[CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/)。

In [ ]:
lab.close()